In [1]:
import pandas as pd 
import numpy as np
from pathlib import Path

# --- Settings ---
INPUT_XLSX = "DATA_UFM_combined_TEST_AREA.xlsx"
SHEET_NAME = "Sheet1"
OUTPUT_XLSX = "DATA_UFM_combined_TEST_AREA_filled_V2.xlsx"   # final + only output

METRICS = [
    "branche_n","branchepct1","branchepct2","branchepct3","branchepct4",
    "branchetxt1","branchetxt2","branchetxt3","branchetxt4","offt_privat",
    "ledighed_nyudd_n","ledighed_nyudd","maanedloen_nyudd_n","maanedloen_nyudd",
    "maanedloenp25_nyudd","maanedloenp75_nyudd","maanedloen_nyudd_aggr",
    "ledighed_10aar_n","ledighed_10aar","maanedloen_10aar_n","maanedloen_10aar",
    "maanedloenp25_10aar","maanedloenp75_10aar","maanedloen_10aar_aggr",
    "arbejdsform_n","arbejdsform_fastedagtimer_pct","arbejdsform_faste_aften_nat_pct",
    "arbejdsform_fleksible_pct","arbejdsform_skiftende_pct","arbejdsform_tilretteselv_pct",
    "arbejdsform_andet_pct","arbejdsform_andet_pct2","arbejdstid_n","arbejdstid_timer",
    "relevans_overens_udd_job_n","relevans_overens_udd_job_ntotal","relevans_overens_udd_job_likert",
    "ruster_til_job_n","ruster_til_job_ntotal","ruster_til_job_likert",
    "udlandet_n","udlandet_dk25_pct","udlandet_dk_pct","udlandet_udlandet_pct",
]

REF_COLS = ["hyppigsteid1", "hyppigsteid2", "hyppigsteid3"]


def norm_udbud(x):
    try:
        return str(int(x))
    except Exception:
        return str(x)

def parse_ref(s):
    if pd.isna(s):
        return None
    s = str(s)
    if ":" in s:
        a, u = s.split(":", 1)
        return (a.strip(), norm_udbud(u.strip()))
    return None


# --- Load ---
df = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_NAME)
df["artikel_id"] = df["artikel_id"].astype(str)
df["udbud_id_str"] = df["udbud_id"].apply(norm_udbud)
df_indexed = df.set_index(["artikel_id", "udbud_id_str"])

# Pre-parse references
ref_tuples_per_row = [
    [t for t in (parse_ref(row[c]) for c in REF_COLS) if t is not None]
    for _, row in df.iterrows()
]

# --- Fill metrics from reference rows ---
df_filled = df.copy()

for i, tuples in enumerate(ref_tuples_per_row):
    existing = [df_indexed.loc[t] for t in tuples if t in df_indexed.index]
    if not existing:
        continue
    ref_df = pd.DataFrame(existing)

    for m in METRICS:
        if m in df_filled.columns and pd.isna(df_filled.at[i, m]):
            avg_val = pd.to_numeric(ref_df[m], errors="coerce").mean(skipna=True)
            if not np.isnan(avg_val):
                df_filled.at[i, m] = avg_val

# --- Create kandidat links ---
df_filled["kandidat_titler"] = ""
df_filled["kandidat_refs"] = ""

idx = df_filled.set_index(["artikel_id", "udbud_id_str"])

for i, tuples in enumerate(ref_tuples_per_row):
    titles = []
    refs = []
    for t in tuples:
        if t in idx.index:
            row = idx.loc[t]
            if "Kandidat" in str(row["displaydocclass"]):
                title = str(row["titel"]).strip()
                key = f"{t[0]}:{t[1]}"
                if title not in titles:
                    titles.append(title)
                if key not in refs:
                    refs.append(key)
    df_filled.at[i, "kandidat_titler"] = " | ".join(titles)
    df_filled.at[i, "kandidat_refs"] = " | ".join(refs)

# ----------------------------------------------------------------------
# NEW: Fill national (udbud_id == 999999) 'optagne' as sum of providers
# ----------------------------------------------------------------------
if "optagne" in df_filled.columns:
    # robust numeric conversion
    df_filled["optagne_num"] = pd.to_numeric(df_filled["optagne"], errors="coerce")

    # sum across rows with same title EXCLUDING national rows
    by_title_sum = (
        df_filled.loc[df_filled["udbud_id_str"] != "999999"]
        .groupby("titel", dropna=False)["optagne_num"]
        .sum(min_count=1)  # NaN if all are NaN
    )

    # write the totals into the national rows (overwrite or fill)
    nat_mask = df_filled["udbud_id_str"] == "999999"
    df_filled.loc[nat_mask, "optagne_num"] = df_filled.loc[nat_mask, "titel"].map(by_title_sum)

    # keep original column name
    df_filled["optagne"] = df_filled["optagne_num"]
    df_filled.drop(columns=["optagne_num"], inplace=True)
# ----------------------------------------------------------------------

# --- Save ONLY ONE OUTPUT ---
df_filled.to_excel(OUTPUT_XLSX, index=False)
print(f"✅ Done! Saved combined output to {OUTPUT_XLSX}")


✅ Done! Saved combined output to DATA_UFM_combined_TEST_AREA_filled_V2.xlsx


NOTE that each variable should reach around 450 the script also struggles with strings as can be seen in the branchetxt1 and so on but that variable might not be that usefull 